# nvMolKit + Nemotron

This guided notebook combines the [nvMolKit repository](https://github.com/NVIDIA-BioNeMo/nvMolKit), [nvMolKit documentation](https://nvidia-bionemo.github.io/nvMolKit/), and the exact [BioNeMo Agent Toolkit nvMolKit skill](https://github.com/NVIDIA-BioNeMo/bionemo-agent-toolkit/blob/main/library-skills/nvMolKit/SKILL.md).

The BioNeMo Agent Toolkit skill informs this notebook's bounded tool contract with the nvMolKit API entry-point map, runtime requirements, recipes, and boundaries. The notebook, not the model, executes the allow-listed Python function.

**Roles.** Brev provides the GPU VM and Secure Link to this notebook. Hosted Nemotron requests one bounded tool call and explains the compact result summary. The notebook validates the five arguments and executes nvMolKit GPU chemistry locally. RDKit parses molecules and prepares displays.

The workflow is a good fit for batched fingerprint, similarity, clustering, conformer, and force-field operations. It is a research demonstration, not a benchmark or a validated scientific study. Its outputs are computational descriptors and candidate geometries; they require task-specific and experimental validation before scientific use.

## 1. Preflight

Resolve the repository from either supported kernel location, import the fixed workflow, request the API key through a hidden notebook prompt when needed, require CUDA, and run a three-molecule GPU probe.

In [ ]:
import os
import sys
from getpass import getpass
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "demo_agent.py").is_file() and (cwd / "data").is_dir():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "demo_agent.py").is_file():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Run this notebook from the repository root or its notebooks/ directory.")

sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "sample_molecules.csv"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py3Dmol
import seaborn as sns
import torch
from IPython.display import Markdown, display
from openai import APIError
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.rdDistGeom import ETKDGv3
from rdkit.Geometry import Point3D

from demo_agent import request_explanation, request_tool_call
import nvmolkit
from nvmolkit.clustering import fused_butina
from nvmolkit.embedMolecules import EmbedMolecules
from nvmolkit.fingerprints import MorganFingerprintGenerator
from nvmolkit.mmffOptimization import MMFFOptimizeMoleculesConfs
from nvmolkit.similarity import crossTanimotoSimilarity
from nvmolkit.types import CoordinateOutput

assert torch.cuda.is_available(), "A CUDA-capable NVIDIA GPU is required."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch/CUDA:", torch.__version__, torch.version.cuda)
print("nvMolKit:", nvmolkit.__version__)

probe = [Chem.MolFromSmiles(smiles) for smiles in ("CCO", "CCN", "c1ccccc1")]
probe_fingerprints = MorganFingerprintGenerator(radius=2, fpSize=1024).GetFingerprints(probe)
torch.cuda.synchronize()
assert tuple(probe_fingerprints.torch().shape) == (3, 32)
print("GPU fingerprint probe passed:", tuple(probe_fingerprints.torch().shape))

## 2. Molecular sample

Parse the bundled sample, exclude and visibly report invalid SMILES, then preview only 24 molecules so the display stays compact.

In [ ]:
sample_raw = pd.read_csv(DATA_PATH)
parsed = [Chem.MolFromSmiles(str(smiles)) for smiles in sample_raw["smiles"]]
valid_mask = np.array([mol is not None for mol in parsed], dtype=bool)
invalid_count = int((~valid_mask).sum())

if invalid_count:
    display(Markdown(f"**Warning:** excluded {invalid_count} invalid molecule(s) before computation."))
else:
    display(Markdown("**Invalid-molecule check:** 0 invalid molecules; none excluded."))

sample = sample_raw.loc[valid_mask].reset_index(drop=True).copy()
mols = [mol for mol in parsed if mol is not None]
assert len(mols) == 256, f"Expected 256 valid bundled molecules, found {len(mols)}."
print(f"Valid molecules: {len(mols)}; excluded invalid molecules: {invalid_count}")

preview_count = min(24, len(mols))
display(Draw.MolsToGridImage(
    mols[:preview_count],
    legends=sample["molecule_id"].iloc[:preview_count].tolist(),
    molsPerRow=6,
    subImgSize=(220, 180),
))

## 3. Nemotron tool call

Force one `analyze_molecule_library` request from hosted Nemotron and validate its five arguments before any chemistry executes. The environment credential is passed directly and never displayed. Authentication failures stop with hosted-key guidance; other hosted call or response failures use a clearly labeled deterministic validated fallback.

In [ ]:
api_key = os.environ.get("NVIDIA_API_KEY", "").strip()
if not api_key:
    api_key = getpass("Hosted NVIDIA Developer API key from the Nemotron build.nvidia.com model page (starts with nvapi-; bare key only; input hidden): ").strip()
    if not api_key:
        raise ValueError("NVIDIA API key is required.")
model = "nvidia/nemotron-3-nano-30b-a3b"
decision = request_tool_call(api_key, model=model)
display(Markdown(f"**Tool-call source:** `{decision.source}`"))
if decision.source == "default_after_error":
    display(Markdown("**Fallback:** Nemotron tool selection failed; using the labeled validated deterministic default."))
    display({"tool_call_error": decision.error})

plan = decision.plan
display(Markdown(f"**Requested tool:** `{decision.tool_name}`"))
display(Markdown("**Validated arguments:**"))
display(plan.model_dump())

## 4. Fingerprints, similarity, and clusters

Define the single local executor that computes Morgan fingerprints, all-pairs Tanimoto similarity, and fused Butina clusters on the GPU. The heatmap is ordered by cluster membership. Defining the function performs no scientific computation.

In [ ]:
def analyze_molecule_library(mols, plan):
    fingerprint_generator = MorganFingerprintGenerator(
        radius=plan.fingerprint_radius,
        fpSize=plan.fingerprint_size,
    )
    fingerprints = fingerprint_generator.GetFingerprints(mols)
    similarity_result = crossTanimotoSimilarity(fingerprints)
    clusters, cluster_sizes = fused_butina(
        fingerprints.torch(), cutoff=plan.cluster_cutoff
    )
    torch.cuda.synchronize()

    similarity = similarity_result.torch().cpu().numpy()
    cluster_ids = [-1] * len(mols)
    for cluster_id, members in enumerate(clusters):
        for molecule_index in members:
            cluster_ids[int(molecule_index)] = cluster_id

    assert similarity.shape == (256, 256)
    assert np.isfinite(similarity).all()
    assert len(cluster_ids) == len(mols) and all(
        cluster_id >= 0 for cluster_id in cluster_ids
    )
    sample["cluster"] = cluster_ids
    print(f"Assigned {len(mols)} molecules to {sample['cluster'].nunique()} clusters.")

    cluster_order = sample.sort_values("cluster", kind="stable").index.to_numpy()
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        similarity[np.ix_(cluster_order, cluster_order)],
        cmap="viridis",
        vmin=0,
        vmax=1,
        cbar_kws={"label": "Tanimoto similarity"},
    )
    plt.title("Morgan fingerprint Tanimoto similarity, ordered by Butina cluster")
    plt.xlabel("Molecules ordered by Butina cluster")
    plt.ylabel("Molecules ordered by Butina cluster")
    plt.tight_layout()
    plt.show()

    eligible = []
    for molecule_index, mol in enumerate(mols):
        hydrogenated = Chem.AddHs(Chem.Mol(mol))
        if AllChem.MMFFHasAllMoleculeParams(hydrogenated):
            eligible.append({
                "molecule_index": molecule_index,
                "cluster": cluster_ids[molecule_index],
                "heavy_atoms": mol.GetNumHeavyAtoms(),
            })

    eligible_frame = pd.DataFrame(
        eligible, columns=["molecule_index", "cluster", "heavy_atoms"]
    ).sort_values(
        ["heavy_atoms", "cluster", "molecule_index"], kind="stable"
    )
    requested_count = min(int(plan.representative_count), 6)
    representative_rows = eligible_frame.drop_duplicates("cluster").head(
        requested_count
    )
    representative_indices = representative_rows["molecule_index"].astype(int).tolist()
    representative_ids = sample.loc[representative_indices, "molecule_id"].tolist()
    representatives = [
        Chem.AddHs(Chem.Mol(mols[index])) for index in representative_indices
    ]

    if len(representatives) < requested_count:
        display(Markdown(
            f"**Selection notice:** requested {requested_count} representatives, but only "
            f"{len(representatives)} distinct-cluster molecules passed the MMFF94 parameter check."
        ))
    else:
        print(
            f"Selected {len(representatives)} MMFF94-eligible representatives "
            "from distinct clusters."
        )
    assert representatives, "No MMFF94-eligible representative molecules were found."
    assert len(representatives) <= 6

    embedding_params = ETKDGv3()
    embedding_params.useRandomCoords = True
    embedding_params.randomSeed = 7
    requested_per_representative = int(plan.conformers_per_representative)
    requested_conformers = len(representatives) * requested_per_representative
    EmbedMolecules(
        representatives,
        embedding_params,
        confsPerMolecule=plan.conformers_per_representative,
        maxIterations=-1,
    )
    generated_counts = [mol.GetNumConformers() for mol in representatives]
    generated_conformers = sum(generated_counts)
    for molecule_id, generated_count in zip(representative_ids, generated_counts):
        if generated_count == 0:
            display(Markdown(
                f"**ETKDGv3 failure — {molecule_id}:** generated zero of "
                f"{requested_per_representative} requested conformers; excluded from MMFF94."
            ))
        elif generated_count < requested_per_representative:
            display(Markdown(
                f"**ETKDGv3 partial result — {molecule_id}:** generated "
                f"{generated_count} of {requested_per_representative} requested "
                "conformers; continuing with those generated."
            ))

    embedded_representatives = [
        (molecule_id, mol)
        for molecule_id, mol, generated_count in zip(
            representative_ids, representatives, generated_counts
        )
        if generated_count > 0
    ]
    if not embedded_representatives:
        raise RuntimeError(
            f"ETKDGv3 generated zero conformers for all {len(representatives)} "
            "representatives; MMFF94 cannot run."
        )
    representative_ids = [molecule_id for molecule_id, _ in embedded_representatives]
    representatives = [mol for _, mol in embedded_representatives]
    optimization_result = MMFFOptimizeMoleculesConfs(
        representatives, maxIters=500, output=CoordinateOutput.DEVICE
    )
    torch.cuda.synchronize()

    assert optimization_result.n_mols == len(representatives)
    assert optimization_result.energies is not None
    assert optimization_result.converged is not None
    assert optimization_result.mol_indices is not None
    assert optimization_result.conf_indices is not None

    energy_values = np.asarray(
        optimization_result.energies.numpy(), dtype=float
    ).reshape(-1)
    convergence_values = np.asarray(
        optimization_result.converged.numpy(), dtype=np.int8
    ).reshape(-1)
    mol_indices = np.asarray(
        optimization_result.mol_indices.numpy(), dtype=int
    ).reshape(-1)
    conf_indices = np.asarray(
        optimization_result.conf_indices.numpy(), dtype=int
    ).reshape(-1)
    coordinate_tensors = optimization_result.per_molecule()
    result_count = len(energy_values)
    mmff_attempted_conformers = result_count

    assert result_count == sum(mol.GetNumConformers() for mol in representatives)
    assert len(convergence_values) == len(mol_indices) == len(conf_indices) == result_count
    assert np.isfinite(energy_values).all()
    assert set(np.unique(convergence_values)).issubset({0, 1})
    assert len(coordinate_tensors) == len(representatives)

    optimization_records = [[] for _ in representatives]
    coordinate_offsets = [0] * len(representatives)
    seen_conformers = set()
    for energy, converged, mol_index, conf_index in zip(
        energy_values, convergence_values, mol_indices, conf_indices
    ):
        mol_index = int(mol_index)
        conf_index = int(conf_index)
        assert 0 <= mol_index < len(representatives)
        assert 0 <= conf_index < representatives[mol_index].GetNumConformers()
        assert (mol_index, conf_index) not in seen_conformers
        seen_conformers.add((mol_index, conf_index))
        coordinate_index = coordinate_offsets[mol_index]
        assert coordinate_index < len(coordinate_tensors[mol_index])
        coordinates = (
            coordinate_tensors[mol_index][coordinate_index].detach().cpu().numpy()
        )
        coordinate_offsets[mol_index] += 1
        mol = representatives[mol_index]
        assert coordinates.shape == (mol.GetNumAtoms(), 3)
        assert np.isfinite(coordinates).all()
        conformer = mol.GetConformer(conf_index)
        assert conformer.GetNumAtoms() == len(coordinates)
        for atom_index, (x, y, z) in enumerate(coordinates):
            conformer.SetAtomPosition(
                atom_index, Point3D(float(x), float(y), float(z))
            )
        optimization_records[mol_index].append({
            "conf_index": conf_index,
            "energy_kcal_per_mol": float(energy),
            "converged": bool(converged == 1),
        })

    assert coordinate_offsets == [len(items) for items in coordinate_tensors]
    assert [len(records) for records in optimization_records] == [
        mol.GetNumConformers() for mol in representatives
    ]
    converged_conformer_count = sum(
        record["converged"]
        for records in optimization_records
        for record in records
    )
    unconverged_count = result_count - converged_conformer_count
    display(Markdown(
        f"**MMFF94 convergence:** {converged_conformer_count} of {result_count} "
        f"conformers converged; {unconverged_count} did not converge within "
        "500 iterations."
    ))

    views = []
    for molecule_id, mol, records in zip(
        representative_ids, representatives, optimization_records
    ):
        converged_records = [record for record in records if record["converged"]]
        if not converged_records:
            display(Markdown(
                f"**{molecule_id}:** no conformer converged within 500 MMFF94 "
                "iterations; no minimized structure is displayed."
            ))
            continue
        best = min(
            converged_records, key=lambda record: record["energy_kcal_per_mol"]
        )
        best_conf = best["conf_index"]
        best_energy = best["energy_kcal_per_mol"]
        mol_block = Chem.MolToMolBlock(mol, confId=best_conf)
        view = py3Dmol.view(width=360, height=280)
        view.addModel(mol_block, "mol")
        view.setStyle({"stick": {}})
        view.zoomTo()
        display(Markdown(
            f"**{molecule_id}** — lowest converged computed MMFF94 energy: "
            f"{best_energy:.2f} kcal/mol. Values are compared only among "
            "conformers of this molecule."
        ))
        view.show()
        views.append(view)

    assert len(views) <= 6
    summary = {
        "molecules": len(mols),
        "clusters": int(sample["cluster"].nunique()),
        "representatives": len(representatives),
        "requested_conformers": requested_conformers,
        "generated_conformers": generated_conformers,
        "mmff_attempted_conformers": mmff_attempted_conformers,
        "converged_conformers": int(converged_conformer_count),
        "fingerprint_method": "Morgan",
        "similarity_method": "Tanimoto",
        "clustering_method": "fused Butina",
        "geometry_method": "ETKDGv3 followed by MMFF94",
    }
    return summary

## 5. Conformers and MMFF94

Select structurally conservative representatives from distinct clusters. Eligibility requires RDKit MMFF parameters on a hydrogenated copy; lower heavy-atom count is preferred. No UFF substitution is made. For each selected molecule, display only its lowest computed MMFF94-energy conformer.

## 6. What the results mean

Summarize only the completed computations and ask hosted Nemotron for a concise interpretation. Explanation failure does not invalidate the already completed deterministic computations.

In [ ]:
if decision.source != "nemotron":
    raise RuntimeError(
        "Scientific tool was not executed because Nemotron did not return a valid tool call."
    )
if decision.tool_name != "analyze_molecule_library":
    raise ValueError(f"Tool is not allow-listed: {decision.tool_name}")
summary = analyze_molecule_library(mols, plan)
display(summary)

SCIENTIFIC_BOUNDARY = (
    "**Scientific boundary:** Computed fingerprints, similarity, clusters, and force-field "
    "geometries are not evidence of binding, activity, ADMET, efficacy, safety, "
    "synthesizability, or clinical relevance. These are not experimentally validated conformations."
)
display(Markdown(SCIENTIFIC_BOUNDARY))
display(Markdown("**Agent-generated interpretation; verify independently.**"))

try:
    explanation = request_explanation(api_key, decision, summary, model=model)
    display(Markdown(explanation))
except (APIError, RuntimeError, ValueError, IndexError, AttributeError) as exc:
    display(Markdown(
        "**Explanation unavailable:** computation succeeded, but the Nemotron explanation failed. "
        f"Error: `{exc}`"
    ))

display(Markdown(
    "**Scientific boundary:** Computed fingerprints, similarity, clusters, and force-field "
    "geometries are not evidence of binding, activity, ADMET, efficacy, safety, "
    "synthesizability, or clinical relevance. These are not experimentally validated conformations."
))